In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from ifc.config import load_data_config
from ifc.config import resolve_project_root

In [ ]:
data_cfg = load_data_config()
train_path, test_path = data_cfg.resolve_paths()
print(train_path)

In [ ]:
df = pd.read_csv(train_path)
df = df.sort_values([data_cfg.id_col, data_cfg.time_col]).copy()
df.shape

3.	TARGET DISTRIBUTION — HEAVY-TAIL VIEW (TRANSFORM-FRIENDLY)

The target variable `revenue_change` is defined as a "year over year" percentage change. Its distribution is often heavy-tailed: extreme values can arise not only from large absolute changes in revenue, but also from very small prior-year values in the denominator. To obtain a readable view of the full distribution while preserving sign information, we apply the signed-log transform:

$$
y_{\mathrm{slog}} = \mathrm{sign}(y)\,\log\left(1 + \left|y\right|\right)
$$

In [ ]:
y = df[data_cfg.target_col].dropna()
# Signed-log transform: preserves the sign while compressing extreme magnitudes
y_slog = np.sign(y) * np.log1p(np.abs(y))

plt.figure()
plt.hist(y_slog, bins=60)
plt.title("Signed-log target distribution: sign(y) * log1p(|y|)")
plt.xlabel("signed-log(revenue_change)")
plt.ylabel("count")
plt.show()

# Numeric evidence (original scale)
y.describe(percentiles=[0.01, 0.05, 0.50, 0.95, 0.99])

The signed-log histogram confirms a heavy-tailed target distribution with extreme events on both sides.
This has two direct implications for the forecasting task:
1.  Visualizations should not rely solely on raw-scale plots
2.  Downstream modeling should consider robustness to       outliers (e.g. robust losses or training-time target clipping)

4.	TEMPORAL DRIFT IN TARGET (2019–2020–2021) (COVID V-SHAPE)

We evaluate whether `revenue_change` exhibits a year-dependent shift in distribution, focusing on fiscal years 2019, 2020 and 2021.
We provide:
1. A year-level summary table with count, mean, median and tail percentiles (p95, p99).
2. A boxplot by year

ATTENTION: for the boxplot only, we clip values to the 1%–99% range to improve readability. This is a visualization convenience and must not be interpreted as removing outliers from the dataset.

In [1]:
years = [2019, 2020, 2021]
target = data_cfg.target_col
time_col = data_cfg.time_col
df_3y = df.loc[df[time_col].isin(years), [time_col, target]].dropna().copy()

summary = (
    df_3y.groupby(time_col)[target]
    .agg(
        n="count",
        mean="mean",
        median="median",
        p95=lambda s: s.quantile(0.95),
        p99=lambda s: s.quantile(0.99),
    )
    .reset_index()
    .sort_values(time_col)
)
summary

NameError: name 'data_cfg' is not defined

In [ ]:
# Clip only for visualization (1%–99%) to avoid the plot being dominated by extreme outliers
clip_low, clip_high = df_3y[target].quantile([0.01, 0.99])
df_3y["target_clip"] = df_3y[target].clip(clip_low, clip_high)

plt.figure()
data_to_plot = [
    df_3y.loc[df_3y[time_col] == yr, "target_clip"].values
    for yr in years
]
plt.boxplot(data_to_plot, labels=[str(y) for y in years], showfliers=False)
plt.title("Revenue_change by fiscal year (clipped 1%–99% for readability)")
plt.xlabel("fiscal_year")
plt.ylabel("revenue_change (clipped)")
plt.show()

The year-level summaries and boxplots typically reveal a clear shift in 2020 followed by a rebound in 2021, consistent with a “V-shape” pattern. This confirms meaningful temporal drift in the target distribution and supports a strictly time-based validation design for the forecasting task.
